In [1]:
df = spark.sql("SELECT * FROM supply_guard.dbo.Silver_inventory LIMIT 1000")
display(df)

StatementMeta(, 88f66c13-4844-4aac-b6ad-d1969389f8ee, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8656ae84-8cac-441a-be41-fa881a09c451)

In [4]:
from pyspark.sql.functions import col

inventory_df = spark.read.table("supply_guard.dbo.Silver_inventory")

gold_inventory = inventory_df.withColumn(
    "InventoryValue",
    col("onHand") * col("Price")
)

gold_inventory.write.mode("overwrite").saveAsTable(
    "supply_guard.dbo.gold_inventory_optimization"
)

print("Gold Inventory Created")

StatementMeta(, 88f66c13-4844-4aac-b6ad-d1969389f8ee, 6, Finished, Available, Finished, False)

Gold Inventory Created


In [5]:
vendor_df = spark.read.table(
    "supply_guard.dbo.silver_vendor_sales_summary"
)

gold_vendor = vendor_df.select(
    "VendorNumber",
    "VendorName",
    "Brand",
    "GrossProfit",
    "ProfitMargin",
    "StockTurnover",
    "SalestoPurchaseRatio",
    "TotalSalesDollars",
    "TotalPurchaseDollars"
)

gold_vendor.write.mode("overwrite").saveAsTable(
    "supply_guard.dbo.gold_vendor_performance"
)

print("Gold Vendor Performance Created")

StatementMeta(, 88f66c13-4844-4aac-b6ad-d1969389f8ee, 7, Finished, Available, Finished, False)

Gold Vendor Performance Created


In [17]:
from pyspark.sql.functions import sum, countDistinct

# Load tables
purchase_df = spark.read.table("supply_guard.dbo.silver_purchase")
invoice_df = spark.read.table("supply_guard.dbo.silver_vendor_invoice")

# Purchase Summary
purchase_summary = purchase_df.groupBy(
    "VendorNumber",
    "VendorName"
).agg(
    sum("Dollars").alias("TotalPurchaseSpend"),
    sum("Quantity").alias("TotalPurchaseQuantity"),
    countDistinct("PONumber").alias("TotalPOs")
)

# Invoice Summary
invoice_summary = invoice_df.groupBy(
    "VendorNumber"
).agg(
    sum("Freight").alias("TotalFreight"),
    countDistinct("InvoiceDate").alias("InvoiceCount")
)

# Join Aggregated Tables
gold_procurement = purchase_summary.join(
    invoice_summary,
    "VendorNumber",
    "left"
)

# Save Gold Table
gold_procurement.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("supply_guard.dbo.gold_procurement_intelligence")

print("Gold Procurement Created")

StatementMeta(, 88f66c13-4844-4aac-b6ad-d1969389f8ee, 19, Finished, Available, Finished, False)

Gold Procurement Created


In [21]:
vendor_dim = spark.read.table(
    "supply_guard.dbo.silver_vendor_sales_summary"
).select(
    "VendorNumber",
    "VendorName"
).dropDuplicates()

vendor_dim.write \
    .mode("overwrite") \
    .saveAsTable("supply_guard.dbo.dim_vendor")

StatementMeta(, 88f66c13-4844-4aac-b6ad-d1969389f8ee, 23, Finished, Available, Finished, False)

In [22]:
product_dim = spark.read.table(
    "supply_guard.dbo.silver_purchase_prices"
).select(
    "Brand",
    "Description",
    "Size",
    "Classification"
).dropDuplicates()

product_dim.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("supply_guard.dbo.dim_product")

StatementMeta(, 88f66c13-4844-4aac-b6ad-d1969389f8ee, 24, Finished, Available, Finished, False)

In [23]:
store_dim = spark.read.table(
    "supply_guard.dbo.silver_inventory"
).select(
    "Store",
    "City"
).dropDuplicates()

store_dim.write \
    .mode("overwrite") \
    .saveAsTable("supply_guard.dbo.dim_store")

StatementMeta(, 88f66c13-4844-4aac-b6ad-d1969389f8ee, 25, Finished, Available, Finished, False)

In [24]:
from pyspark.sql.functions import explode, sequence, to_date, lit

date_dim = spark.sql("""
SELECT explode(
    sequence(
        to_date('2015-01-01'),
        to_date('2025-12-31'),
        interval 1 day
    )
) as Date
""")

date_dim.createOrReplaceTempView("date_dim")

date_dim = spark.sql("""
SELECT
    Date,
    year(Date) as Year,
    quarter(Date) as Quarter,
    month(Date) as Month,
    day(Date) as Day
FROM date_dim
""")

date_dim.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("supply_guard.dbo.dim_date")

StatementMeta(, 88f66c13-4844-4aac-b6ad-d1969389f8ee, 26, Finished, Available, Finished, False)

In [25]:
from pyspark.sql.functions import col

fact_inventory = spark.read.table(
    "supply_guard.dbo.silver_inventory"
).withColumn(
    "InventoryValue",
    col("onHand") * col("Price")
)

fact_inventory.write \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable("supply_guard.dbo.fact_inventory")

StatementMeta(, 88f66c13-4844-4aac-b6ad-d1969389f8ee, 27, Finished, Available, Finished, False)

In [26]:
fact_vendor = spark.read.table(
    "supply_guard.dbo.silver_vendor_sales_summary"
)

fact_vendor.write \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable("supply_guard.dbo.fact_vendor_performance")

StatementMeta(, 88f66c13-4844-4aac-b6ad-d1969389f8ee, 28, Finished, Available, Finished, False)

In [27]:
from pyspark.sql.functions import sum,countDistinct

purchase_df = spark.read.table(
    "supply_guard.dbo.silver_purchase"
)

invoice_df = spark.read.table(
    "supply_guard.dbo.silver_vendor_invoice"
)

purchase_summary = purchase_df.groupBy(
    "VendorNumber",
    "VendorName"
).agg(
    sum("Dollars").alias("TotalPurchaseSpend"),
    sum("Quantity").alias("TotalPurchaseQuantity"),
    countDistinct("PONumber").alias("TotalPOs")
)

invoice_summary = invoice_df.groupBy(
    "VendorNumber"
).agg(
    sum("Freight").alias("TotalFreight"),
    countDistinct("InvoiceDate").alias("InvoiceCount")
)

fact_procurement = purchase_summary.join(
    invoice_summary,
    "VendorNumber",
    "left"
)

fact_procurement.write \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable("supply_guard.dbo.fact_procurement")

StatementMeta(, 88f66c13-4844-4aac-b6ad-d1969389f8ee, 29, Finished, Available, Finished, False)

In [1]:
# Read source tables
inventory = spark.read.table("supply_guard.dbo.Silver_inventory")
purchase = spark.read.table("supply_guard.dbo.Silver_purchase")

# Delete old tables if exist
spark.sql("DROP TABLE IF EXISTS supply_guard.dbo.dim_store")
spark.sql("DROP TABLE IF EXISTS supply_guard.dbo.dim_vendor")

# Create clean dim_store
dim_store = (
    inventory
    .select("Store", "City")
    .filter("City IS NOT NULL")
    .dropDuplicates(["Store"])
)

# Create clean dim_vendor
dim_vendor = (
    purchase
    .select("VendorNumber", "VendorName")
    .dropDuplicates(["VendorNumber"])
)

# Save new dimensions
dim_store.write \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("supply_guard.dbo.dim_store")

dim_vendor.write \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("supply_guard.dbo.dim_vendor")

print("dim_store and dim_vendor recreated successfully")

StatementMeta(, 6fc374d6-d779-47e9-b502-ad21fa9dae4c, 3, Finished, Available, Finished, False)

dim_store and dim_vendor recreated successfully


In [1]:
   spark.sql("show tables").show(50, False)

StatementMeta(, 112389cd-69db-45f0-bf95-33c3c9cd2bd3, 3, Finished, Available, Finished, False)

+-------------------------------+-----------------------------+-----------+
|namespace                      |tableName                    |isTemporary|
+-------------------------------+-----------------------------+-----------+
|`My workspace`.supply_guard.dbo|Bronze_begin_inventory       |false      |
|`My workspace`.supply_guard.dbo|Bronze_end_inventory         |false      |
|`My workspace`.supply_guard.dbo|Bronze_purchase              |false      |
|`My workspace`.supply_guard.dbo|Bronze_purchase_prices       |false      |
|`My workspace`.supply_guard.dbo|Bronze_sales                 |false      |
|`My workspace`.supply_guard.dbo|Bronze_vendor_invoice        |false      |
|`My workspace`.supply_guard.dbo|Bronze_vendor_sales_summary  |false      |
|`My workspace`.supply_guard.dbo|Silver_begin_inventory       |false      |
|`My workspace`.supply_guard.dbo|Silver_end_inventory         |false      |
|`My workspace`.supply_guard.dbo|Silver_inventory             |false      |
|`My workspa

In [2]:
   spark.table("Silver_purchase_prices").printSchema()
   spark.table("dim_product").printSchema()
   p = spark.table("dim_product")
   print("rows:", p.count(), "| distinct Brand:", p.select("Brand").distinct().count())

StatementMeta(, 112389cd-69db-45f0-bf95-33c3c9cd2bd3, 4, Finished, Available, Finished, False)

root
 |-- Brand: long (nullable = true)
 |-- Description: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Size: string (nullable = true)
 |-- Volume: long (nullable = true)
 |-- Classification: long (nullable = true)
 |-- PurchasePrice: double (nullable = true)
 |-- VendorNumber: long (nullable = true)
 |-- VendorName: string (nullable = true)

root
 |-- Brand: long (nullable = true)
 |-- Description: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Classification: long (nullable = true)

rows: 12261 | distinct Brand: 12261


In [3]:
   from pyspark.sql import functions as F
   prod = spark.table("dim_product")
   pp = (spark.table("Silver_purchase_prices")
         .select("Brand",
                 F.col("VendorNumber").cast("string").alias("VendorNumber"),
                 F.col("PurchasePrice").cast("double").alias("PurchasePrice"))
         .dropDuplicates(["Brand"]))
   out = prod.join(pp, "Brand", "left")
   print("before:", prod.count(), "after:", out.count(),
         "missing price:", out.filter("PurchasePrice is null").count())
   out.show(5, False)

StatementMeta(, 112389cd-69db-45f0-bf95-33c3c9cd2bd3, 5, Finished, Available, Finished, False)

before: 12261 after: 12261 missing price: 0
+-----+---------------------------+------+--------------+------------+-------------+
|Brand|Description                |Size  |Classification|VendorNumber|PurchasePrice|
+-----+---------------------------+------+--------------+------------+-------------+
|6498 |Franzia Chablis Old World C|5000mL|2             |9815        |9.73         |
|25367|Alexander Vly Vyds Merlot  |1500ml|2             |7153        |26.89        |
|24051|Genbei San Demonslayer Sake|180mL |2             |10754       |3.89         |
|39087|Corbett Canyon Pnt Grigio  |3000mL|2             |9815        |6.66         |
|5872 |Godiva Chocolate Liqueur   |50mL  |1             |3960        |2.32         |
+-----+---------------------------+------+--------------+------------+-------------+
only showing top 5 rows



In [4]:
   out.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim_product")

StatementMeta(, 112389cd-69db-45f0-bf95-33c3c9cd2bd3, 6, Finished, Available, Finished, False)

In [1]:
   from pyspark.sql import functions as F
   p = spark.table("dim_product")
   fixed = p.withColumn("VendorNumber", F.col("VendorNumber").cast("long"))
   print(fixed.count(), fixed.filter("VendorNumber is null").count())
   fixed.printSchema()

StatementMeta(, 7642ad4e-047a-4d5f-b4b8-5251dd419efb, 3, Finished, Available, Finished, False)

12261 0
root
 |-- Brand: long (nullable = true)
 |-- Description: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Classification: long (nullable = true)
 |-- VendorNumber: long (nullable = true)
 |-- PurchasePrice: double (nullable = true)



In [2]:
   fixed.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim_product")

StatementMeta(, 7642ad4e-047a-4d5f-b4b8-5251dd419efb, 4, Finished, Available, Finished, True)

In [1]:
rows = [("Under 60 days", 1), ("60-120 days", 2), ("120-180 days", 3),
        ("Over 180 days", 4), ("No sales", 5)]
spark.createDataFrame(rows, ["Bucket", "BucketOrder"]) \
     .write.mode("overwrite").saveAsTable("dim_health_bucket")
spark.table("dim_health_bucket").show()

StatementMeta(, 2cc80ef1-f7d0-4b73-adca-5acfcdb99198, 3, Finished, Available, Finished, False)

+-------------+-----------+
|       Bucket|BucketOrder|
+-------------+-----------+
|Under 60 days|          1|
|Over 180 days|          4|
|     No sales|          5|
| 120-180 days|          3|
|  60-120 days|          2|
+-------------+-----------+



In [1]:
from pyspark.sql import functions as F
names = {"DIAGEO NORTH AMERICA INC":"Diageo","MARTIGNETTI COMPANIES":"Martignetti","JIM BEAM BRANDS COMPANY":"Jim Beam",
 "PERNOD RICARD USA":"Pernod Ricard","BACARDI USA INC":"Bacardi","CONSTELLATION BRANDS INC":"Constellation",
 "BROWN-FORMAN CORP":"Brown-Forman","ULTRA BEVERAGE COMPANY LLP":"Ultra Beverage","E & J GALLO WINERY":"E&J Gallo",
 "M S WALKER INC":"M.S. Walker","MOET HENNESSEY USA INC":"Moet Hennessy","SAZERAC CO INC":"Sazerac",
 "WILLIAM GRANT & SONS INC":"William Grant"}
m = F.create_map([F.lit(x) for kv in names.items() for x in kv])
fallback = F.initcap(F.regexp_extract(F.trim(F.col("VendorName")), r"^(\S+(?:\s+\S+)?)", 1))
out = (spark.table("dim_vendor").drop("ShortName")
       .withColumn("ShortName", F.coalesce(m[F.trim(F.col("VendorName"))], fallback)))
out.select("VendorName","ShortName").show(15, False)

StatementMeta(, e49fb7b2-b988-4e85-9eb8-027e35842671, 3, Finished, Available, Finished, False)

+-----------------------------+------------------+
|VendorName                   |ShortName         |
+-----------------------------+------------------+
|IRA GOLDMAN AND WILLIAMS, LLP|Ira Goldman       |
|AAPER ALCOHOL & CHEMICAL CO  |Aaper Alcohol     |
|ADAMBA IMPORTS INTL INC      |Adamba Imports    |
|ALTAMAR BRANDS LLC           |Altamar Brands    |
|AMERICAN SPIRITS EXCHANGE    |American Spirits  |
|APPOLO VINEYARDS LLC         |Appolo Vineyards  |
|ATLANTIC IMPORTING COMPANY   |Atlantic Importing|
|BACARDI USA INC              |Bacardi           |
|BANFI PRODUCTS CORP          |Banfi Products    |
|STATE WINE & SPIRITS         |State Wine        |
|SAZERAC NORTH AMERICA INC.   |Sazerac North     |
|BRONCO WINE COMPANY          |Bronco Wine       |
|BROWN-FORMAN CORP            |Brown-Forman      |
|BULLY BOY DISTILLERS         |Bully Boy         |
|BLACK ROCK SPIRITS LLC       |Black Rock        |
+-----------------------------+------------------+
only showing top 15 rows



In [2]:
out.write.mode("overwrite").option("overwriteSchema","true").saveAsTable("dim_vendor")

StatementMeta(, e49fb7b2-b988-4e85-9eb8-027e35842671, 4, Finished, Available, Finished, False)

In [3]:
rows = [
 (1,"Which vendors take most of our spend?","Qualify backup suppliers and shift about 20% of top-3 volume over time.","#FEE2E2","#B91C1C"),
 (2,"Which vendors make the most profit?","Protect these relationships with volume-based agreements.","#DCFCE7","#15803D"),
 (3,"Why do some vendors look unprofitable?","Renegotiate cost prices and terms with the below-average vendors first.","#DCFCE7","#15803D"),
 (4,"Which vendors turn stock fastest?","Cut reorder quantities for the slow vendors.","#FEF3C7","#B45309"),
 (5,"What ties up our inventory capital?","Mark down or return dead stock; lower reorder points on slow items.","#FEF3C7","#B45309"),
 (6,"Which cities hold the most stock?","No rebalancing needed. Monitor quarterly.","#DCFCE7","#15803D"),
 (7,"Do purchase volumes match profitability?","Add margin % and profit per unit to the vendor scorecard.","#DBEAFE","#1D4ED8"),
 (8,"Are freight costs proportionate?","No vendor-specific fix. Review only at contract renewal.","#DCFCE7","#15803D")]
spark.createDataFrame(rows, ["Id","Question","Action","ImpactBg","ImpactFg"]) \
     .write.mode("overwrite").option("overwriteSchema","true").saveAsTable("dim_insight")
spark.table("dim_insight").select("Id","Question").show(truncate=False)

StatementMeta(, e49fb7b2-b988-4e85-9eb8-027e35842671, 5, Finished, Available, Finished, False)

+---+----------------------------------------+
|Id |Question                                |
+---+----------------------------------------+
|1  |Which vendors take most of our spend?   |
|2  |Which vendors make the most profit?     |
|7  |Do purchase volumes match profitability?|
|8  |Are freight costs proportionate?        |
|3  |Why do some vendors look unprofitable?  |
|4  |Which vendors turn stock fastest?       |
|5  |What ties up our inventory capital?     |
|6  |Which cities hold the most stock?       |
+---+----------------------------------------+



In [1]:
from pyspark.sql import functions as F, Window

def put(df, name):
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)

pp = (spark.table("Silver_purchase_prices")
      .select("Brand", F.col("VendorNumber").cast("long").alias("VendorNumber"),
              F.col("PurchasePrice").cast("double").alias("PurchasePrice"))
      .dropDuplicates(["Brand"]))
prod = (spark.table("dim_product").select("Brand", "Description", "Size", "Classification")
        .join(pp, "Brand", "left"))
print("rows:", prod.count(), "| missing price:", prod.filter("PurchasePrice is null").count())
put(prod, "dim_product")

StatementMeta(, 6cc3dd1c-cd09-41fd-aa01-2f5e59001030, 3, Finished, Available, Finished, False)

rows: 12261 | missing price: 0


In [2]:
inv = spark.table("fact_inventory")
cols = [c for c in inv.columns if c != "InventoryCost"]
price = spark.table("dim_product").select("Brand", "PurchasePrice")
inv2 = (inv.select(*cols).join(price, "Brand", "left")
        .withColumn("InventoryCost", F.col("onHand") * F.col("PurchasePrice"))
        .drop("PurchasePrice"))
last = inv2.agg(F.max("SnapshotDate")).first()[0]
print("rows:", inv.count(), inv2.count())
print("closing cost:", inv2.filter(F.col("SnapshotDate") == last).agg(F.sum("InventoryCost")).first()[0])
put(inv2, "fact_inventory")

StatementMeta(, 6cc3dd1c-cd09-41fd-aa01-2f5e59001030, 4, Finished, Available, Finished, False)

rows: 431018 431018
closing cost: 55429569.00000067


In [3]:
fvp = spark.table("fact_vendor_performance")
cols = [c for c in fvp.columns if c != "COGS_Amount"]
fvp2 = fvp.select(*cols).withColumn("COGS_Amount", F.col("TotalSalesQuantity") * F.col("PurchasePrice"))
print("COGS:", fvp2.agg(F.sum("COGS_Amount")).first()[0])
put(fvp2, "fact_vendor_performance")

StatementMeta(, 6cc3dd1c-cd09-41fd-aa01-2f5e59001030, 5, Finished, Available, Finished, False)

COGS: 313151835.9900006


In [4]:
inv = spark.table("fact_inventory")
last = inv.agg(F.max("SnapshotDate")).first()[0]
stock = (inv.filter(F.col("SnapshotDate") == last).groupBy("Brand")
         .agg(F.sum("InventoryCost").alias("ClosingStockCost")))
cogs = (spark.table("fact_vendor_performance").groupBy("Brand")
        .agg(F.sum("COGS_Amount").alias("BrandCOGS")))
prod = spark.table("dim_product")
prod = prod.drop(*[c for c in ["ClosingStockCost","BrandCOGS","DaysOfSupply","HealthOrder"] if c in prod.columns])
p2 = (prod.join(stock, "Brand", "left").join(cogs, "Brand", "left")
      .fillna({"ClosingStockCost": 0.0, "BrandCOGS": 0.0})
      .withColumn("DaysOfSupply", F.when(F.col("BrandCOGS") > 0, F.col("ClosingStockCost") / F.col("BrandCOGS") * 365))
      .withColumn("HealthOrder",
          F.when(F.col("ClosingStockCost") <= 0, F.lit(0))
           .when(F.col("BrandCOGS") <= 0, F.lit(5))
           .when(F.col("DaysOfSupply") < 60, F.lit(1))
           .when(F.col("DaysOfSupply") < 120, F.lit(2))
           .when(F.col("DaysOfSupply") <= 180, F.lit(3))
           .otherwise(F.lit(4))))
p2.groupBy("HealthOrder").agg(F.count("*").alias("brands"), F.sum("ClosingStockCost").alias("stock")).orderBy("HealthOrder").show()
put(p2, "dim_product")

StatementMeta(, 6cc3dd1c-cd09-41fd-aa01-2f5e59001030, 6, Finished, Available, Finished, False)

+-----------+------+--------------------+
|HealthOrder|brands|               stock|
+-----------+------+--------------------+
|          0|  2763|                 0.0|
|          1|  2634|2.0753949920000046E7|
|          2|  2469|1.7143330749999948E7|
|          3|  1121|          6885076.47|
|          4|  2801|1.0228703119999986E7|
|          5|   473|   418508.7399999998|
+-----------+------+--------------------+



In [5]:
names = {"DIAGEO NORTH AMERICA INC":"Diageo","MARTIGNETTI COMPANIES":"Martignetti","JIM BEAM BRANDS COMPANY":"Jim Beam",
 "PERNOD RICARD USA":"Pernod Ricard","BACARDI USA INC":"Bacardi","CONSTELLATION BRANDS INC":"Constellation",
 "BROWN-FORMAN CORP":"Brown-Forman","ULTRA BEVERAGE COMPANY LLP":"Ultra Beverage","E & J GALLO WINERY":"E&J Gallo",
 "M S WALKER INC":"M.S. Walker","MOET HENNESSEY USA INC":"Moet Hennessy","SAZERAC CO INC":"Sazerac",
 "WILLIAM GRANT & SONS INC":"William Grant"}
m = F.create_map([F.lit(x) for kv in names.items() for x in kv])
fallback = F.initcap(F.regexp_extract(F.trim(F.col("VendorName")), r"^(\S+(?:\s+\S+)?)", 1))
v = spark.table("dim_vendor")
v = v.drop(*[c for c in ["ShortName","SpendRank","GPRank","UnitsRank"] if c in v.columns])
v = v.withColumn("ShortName", F.coalesce(m[F.trim(F.col("VendorName"))], fallback))

def rank_of(df, key, val, name):
    return (df.groupBy(key).agg(F.sum(val).alias("v"))
              .withColumn(name, F.rank().over(Window.orderBy(F.col("v").desc())))
              .select(key, name))
sp = rank_of(spark.table("fact_procurement"), "VendorNumber", "TotalPurchaseSpend", "SpendRank")
un = rank_of(spark.table("fact_procurement"), "VendorNumber", "TotalPurchaseQuantity", "UnitsRank")
gp = rank_of(spark.table("fact_vendor_performance"), "VendorNumber", "GrossProfit", "GPRank")
v2 = v.join(sp, "VendorNumber", "left").join(un, "VendorNumber", "left").join(gp, "VendorNumber", "left")
v2.orderBy("SpendRank").select("VendorName","ShortName","SpendRank","GPRank","UnitsRank").show(5, False)
put(v2, "dim_vendor")

StatementMeta(, 6cc3dd1c-cd09-41fd-aa01-2f5e59001030, 7, Finished, Available, Finished, False)

+------------------------+-------------+---------+------+---------+
|VendorName              |ShortName    |SpendRank|GPRank|UnitsRank|
+------------------------+-------------+---------+------+---------+
|DIAGEO NORTH AMERICA INC|Diageo       |1        |1     |1        |
|MARTIGNETTI COMPANIES   |Martignetti  |2        |2     |3        |
|JIM BEAM BRANDS COMPANY |Jim Beam     |3        |5     |2        |
|PERNOD RICARD USA       |Pernod Ricard|4        |4     |6        |
|BACARDI USA INC         |Bacardi      |5        |6     |7        |
+------------------------+-------------+---------+------+---------+
only showing top 5 rows



In [6]:
inv = spark.table("fact_inventory")
last = inv.agg(F.max("SnapshotDate")).first()[0]
city = (inv.filter(F.col("SnapshotDate") == last).groupBy("City").agg(F.sum("InventoryValue").alias("v"))
        .withColumn("CityRank", F.rank().over(Window.orderBy(F.col("v").desc()))).select("City","CityRank"))
st = spark.table("dim_store")
st = st.drop("CityRank") if "CityRank" in st.columns else st
st2 = st.join(city, "City", "left")
st2.orderBy("CityRank").select("City","CityRank").show(3, False)
put(st2, "dim_store")

StatementMeta(, 6cc3dd1c-cd09-41fd-aa01-2f5e59001030, 8, Finished, Available, Finished, False)

+-----------+--------+
|City       |CityRank|
+-----------+--------+
|TYWARDREATH|NULL    |
|MOUNTMEND  |1       |
|MOUNTMEND  |1       |
+-----------+--------+
only showing top 3 rows



In [7]:
for t in ["dim_product","dim_vendor","dim_store","fact_inventory","fact_vendor_performance","fact_procurement"]:
    spark.sql(f"OPTIMIZE {t} VORDER")

StatementMeta(, 6cc3dd1c-cd09-41fd-aa01-2f5e59001030, 9, Finished, Available, Finished, False)

In [1]:
st = spark.table("dim_store")
print("stores without a rank:", st.filter("CityRank is null").count())
st.filter("CityRank is null").select("Store", "City").show(10, False)

StatementMeta(, f2be1b50-9aca-48a0-884c-3d392d89e2d0, 3, Finished, Available, Finished, False)

stores without a rank: 1
+-----+-----------+
|Store|City       |
+-----+-----------+
|46   |TYWARDREATH|
+-----+-----------+



In [1]:
from pyspark.sql import functions as F
(spark.range(1)
    .select(F.date_format(F.from_utc_timestamp(F.current_timestamp(), "Asia/Kolkata"),
                          "dd MMM yyyy, hh:mm a").alias("LastRefreshText"))
    .write.mode("overwrite").saveAsTable("dim_refresh"))
spark.table("dim_refresh").show(truncate=False)

StatementMeta(, fc229672-ba94-4075-b896-a59e1b9c14af, 3, Finished, Available, Finished, False)

+---------------------+
|LastRefreshText      |
+---------------------+
|11 Oct 2026, 08:43 AM|
+---------------------+

